# Phân tích toàn diện bộ dữ liệu tài chính ViFinQA

Notebook này phân tích **toàn bộ** dữ liệu trong bốn tệp Parquet và đối chiếu với `manifest.json`:

- `documents.parquet`: tài liệu nguồn và metadata;
- `tables.parquet`: bảng đã chuẩn hóa;
- `source_table_occurrences.parquet`: dấu vết mỗi bảng trong tài liệu nguồn;
- `issues.parquet`: lỗi/cảnh báo chất lượng ở cấp ô, trường, bảng và tài liệu.

Mục tiêu là kiểm tra quy mô, schema, độ đầy đủ, phân phối, quan hệ khóa, provenance và các điểm nghẽn chất lượng dữ liệu. DuckDB được dùng để quét trực tiếp Parquet, phù hợp với bảng `issues` lớn và tránh nạp toàn bộ dữ liệu vào RAM.

## 1. Thiết lập môi trường và tìm thư mục dữ liệu

Notebook tự tìm dữ liệu trong `./upload`, `./data` hoặc thư mục hiện tại. Có thể đặt biến môi trường `VIFINQA_DATA_DIR` để chỉ định vị trí khác.

In [1]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


d:\GitHub\financial-assistant\.venv\Scripts\python.exe: No module named pip


In [2]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

required = ["duckdb", "pyarrow"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)

candidates = []
if os.getenv("VIFINQA_DATA_DIR"):
    candidates.append(Path(os.environ["VIFINQA_DATA_DIR"]))
candidates.extend([Path("upload"), Path("data"), Path(".")])
required_files = {
    "documents": "documents.parquet",
    "tables": "tables.parquet",
    "occurrences": "source_table_occurrences.parquet",
    "issues": "issues.parquet",
    "manifest": "manifest.json",
}
DATA_DIR = next(
    (p.resolve() for p in candidates if all((p / f).exists() for f in required_files.values())),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ 5 tệp dữ liệu. Hãy đặt VIFINQA_DATA_DIR trỏ tới thư mục chứa dữ liệu."
    )

print(f"Thư mục dữ liệu: {DATA_DIR}")

ModuleNotFoundError: No module named 'seaborn'

## 2. Đọc manifest, đăng ký các view và kiểm kê tệp

In [ ]:
import json

with open(DATA_DIR / required_files["manifest"], encoding="utf-8") as f:
    manifest = json.load(f)

con = duckdb.connect(database=":memory:")
view_files = {
    "documents": required_files["documents"],
    "tables": required_files["tables"],
    "occurrences": required_files["occurrences"],
    "issues": required_files["issues"],
}
for view_name, filename in view_files.items():
    path = (DATA_DIR / filename).as_posix().replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_parquet('{path}')")

def sql(query: str, params=None) -> pd.DataFrame:
    '''Chạy SQL và trả về DataFrame nhỏ đã tổng hợp.'''
    return con.execute(query, params or []).fetchdf()

file_inventory = pd.DataFrame([
    {
        "tệp": filename,
        "kích_thước_MB": round((DATA_DIR / filename).stat().st_size / 1024**2, 3),
        "số_dòng": int(sql(f"SELECT COUNT(*) AS n FROM {view}").iloc[0, 0]),
    }
    for view, filename in view_files.items()
])

display(Markdown(f"**Dataset fingerprint:** `{manifest.get('dataset_fingerprint')}`  "))
display(Markdown(f"**Schema version:** `{manifest.get('schema_version')}`"))
display(file_inventory)

## 3. Schema và mức độ thiếu dữ liệu

In [ ]:
schema_frames = []
for view in view_files:
    info = sql(f"DESCRIBE SELECT * FROM {view}").rename(columns={"column_name": "cột", "column_type": "kiểu"})
    info.insert(0, "bảng", view)
    schema_frames.append(info[["bảng", "cột", "kiểu", "null", "key"]])
schema = pd.concat(schema_frames, ignore_index=True)
display(schema)

In [ ]:
def missingness(view: str) -> pd.DataFrame:
    columns = sql(f"DESCRIBE SELECT * FROM {view}")["column_name"].tolist()
    n = int(sql(f"SELECT COUNT(*) n FROM {view}").iloc[0, 0])
    expressions = ", ".join(
        f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END)::BIGINT AS "{c}"' for c in columns
    )
    counts = sql(f"SELECT {expressions} FROM {view}").iloc[0]
    return pd.DataFrame({
        "bảng": view,
        "cột": columns,
        "số_null": [int(counts[c]) for c in columns],
        "tỷ_lệ_null_%": [round(100 * int(counts[c]) / n, 3) if n else 0 for c in columns],
    })

missing = pd.concat([missingness(v) for v in view_files], ignore_index=True)
display(missing.sort_values(["bảng", "tỷ_lệ_null_%"], ascending=[True, False]))

## 4. Đối chiếu số lượng với manifest

In [ ]:
actual = {
    "document_count": int(sql("SELECT COUNT(*) n FROM documents").iloc[0, 0]),
    "table_count": int(sql("SELECT COUNT(*) n FROM tables").iloc[0, 0]),
    "issue_count": int(sql("SELECT COUNT(*) n FROM issues").iloc[0, 0]),
}
checks = []
for key, value in actual.items():
    expected = manifest.get(key)
    checks.append({"chỉ_tiêu": key, "manifest": expected, "thực_tế": value, "khớp": expected == value})

status_actual = dict(sql("SELECT status, COUNT(*)::BIGINT n FROM occurrences GROUP BY status").values.tolist())
for status, expected in manifest.get("source_table_occurrence_counts", {}).items():
    if status == "total":
        value = sum(status_actual.values())
    else:
        value = int(status_actual.get(status, 0))
    checks.append({
        "chỉ_tiêu": f"occurrences.{status}", "manifest": expected, "thực_tế": value, "khớp": expected == value
    })
display(pd.DataFrame(checks))

## 5. Phân tích tài liệu nguồn

Phần này đo độ phủ công ty–năm–phạm vi báo cáo, kích thước tệp, tình trạng inventory và tính duy nhất của khóa tài liệu.

In [ ]:
doc_summary = sql('''
SELECT
    COUNT(*)::BIGINT AS documents,
    COUNT(DISTINCT doc_id)::BIGINT AS unique_doc_id,
    COUNT(DISTINCT relative_path)::BIGINT AS unique_paths,
    COUNT(DISTINCT sha256)::BIGINT AS unique_sha256,
    COUNT(DISTINCT company_code)::BIGINT AS companies,
    MIN(report_year) AS min_year,
    MAX(report_year) AS max_year,
    ROUND(AVG(file_size_bytes)/1024, 1) AS avg_size_kb,
    ROUND(MEDIAN(file_size_bytes)/1024, 1) AS median_size_kb
FROM documents
''')
display(doc_summary)

coverage = sql('''
SELECT report_year, statement_scope, COUNT(*)::BIGINT AS documents
FROM documents
GROUP BY report_year, statement_scope
ORDER BY report_year, statement_scope
''')
pivot = coverage.pivot(index="report_year", columns="statement_scope", values="documents").fillna(0)
display(pivot.astype(int))
pivot.plot(kind="bar", stacked=True, colormap="tab20c")
plt.title("Số tài liệu theo năm và phạm vi báo cáo")
plt.xlabel("Năm báo cáo")
plt.ylabel("Số tài liệu")
plt.legend(title="Phạm vi", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
company_coverage = sql('''
SELECT company_code,
       COUNT(*)::BIGINT AS documents,
       COUNT(DISTINCT report_year)::BIGINT AS years,
       MIN(report_year) AS first_year,
       MAX(report_year) AS last_year,
       COUNT(DISTINCT statement_scope)::BIGINT AS scopes
FROM documents
GROUP BY company_code
ORDER BY documents DESC, company_code
''')
display(company_coverage.head(20))
display(company_coverage[["documents", "years", "scopes"]].describe().round(2))

## 6. Phân tích các bảng chuẩn hóa

In [ ]:
table_summary = sql('''
SELECT
    COUNT(*)::BIGINT AS tables,
    COUNT(DISTINCT table_id)::BIGINT AS unique_table_id,
    COUNT(DISTINCT doc_id)::BIGINT AS documents_with_tables,
    SUM(row_count)::BIGINT AS total_rows_in_tables,
    SUM(row_count * column_count)::BIGINT AS estimated_cells,
    ROUND(AVG(row_count), 2) AS avg_rows,
    ROUND(MEDIAN(row_count), 2) AS median_rows,
    MAX(row_count) AS max_rows,
    ROUND(AVG(column_count), 2) AS avg_columns,
    MAX(column_count) AS max_columns,
    ROUND(AVG(quality_score), 4) AS avg_quality
FROM tables
''')
display(table_summary)

table_meta = sql('''
SELECT
    SUM(title_raw IS NULL)::BIGINT AS missing_title,
    SUM(statement_type IS NULL)::BIGINT AS missing_statement_type,
    SUM(unit_raw IS NULL)::BIGINT AS missing_unit_raw,
    SUM(unit_normalized IS NULL)::BIGINT AS missing_unit_normalized,
    SUM(csv_path IS NULL)::BIGINT AS missing_csv_path
FROM tables
''')
display(table_meta)

In [ ]:
statement_dist = sql('''
SELECT COALESCE(statement_type, '(thiếu)') AS statement_type, COUNT(*)::BIGINT AS n
FROM tables GROUP BY 1 ORDER BY n DESC
''')
unit_dist = sql('''
SELECT COALESCE(unit_normalized, '(thiếu)') AS unit_normalized, COUNT(*)::BIGINT AS n
FROM tables GROUP BY 1 ORDER BY n DESC
''')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=statement_dist, y="statement_type", x="n", ax=axes[0], color="#4C78A8")
axes[0].set_title("Loại báo cáo của bảng")
axes[0].set_xlabel("Số bảng"); axes[0].set_ylabel("")
sns.barplot(data=unit_dist, y="unit_normalized", x="n", ax=axes[1], color="#F58518")
axes[1].set_title("Đơn vị chuẩn hóa")
axes[1].set_xlabel("Số bảng"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()
display(statement_dist)
display(unit_dist)

In [ ]:
shape_quantiles = sql('''
SELECT
    QUANTILE_CONT(row_count, [0, .25, .5, .75, .9, .95, .99, 1]) AS row_count_quantiles,
    QUANTILE_CONT(column_count, [0, .25, .5, .75, .9, .95, .99, 1]) AS column_count_quantiles
FROM tables
''')
display(shape_quantiles.T.rename(columns={0: "quantiles [min,p25,p50,p75,p90,p95,p99,max]"}))

largest_tables = sql('''
SELECT t.table_id, d.company_code, d.report_year, d.statement_scope,
       t.title_raw, t.row_count, t.column_count,
       t.row_count * t.column_count AS estimated_cells
FROM tables t JOIN documents d USING (doc_id)
ORDER BY estimated_cells DESC
LIMIT 20
''')
display(largest_tables)

## 7. Dấu vết bảng nguồn và kết quả canonical / duplicate / rejected

In [ ]:
occ_status = sql('''
SELECT status, COUNT(*)::BIGINT AS occurrences,
       COUNT(DISTINCT doc_id)::BIGINT AS documents,
       COUNT(DISTINCT canonical_table_id)::BIGINT AS canonical_table_ids
FROM occurrences
GROUP BY status
ORDER BY occurrences DESC
''')
display(occ_status)

rejections = sql('''
SELECT COALESCE(rejection_code, '(không có)') AS rejection_code, COUNT(*)::BIGINT AS n
FROM occurrences
WHERE status = 'rejected'
GROUP BY 1 ORDER BY n DESC
''')
display(rejections)

duplicate_occurrences = sql('''
SELECT source_table_id, doc_id, relative_path, ordinal, canonical_table_id,
       duplicate_of_relative_path
FROM occurrences
WHERE status = 'duplicate'
ORDER BY relative_path, ordinal
''')
display(duplicate_occurrences)

## 8. Phân tích toàn bộ issue chất lượng dữ liệu

In [ ]:
issue_by_code = sql('''
SELECT code,
       COUNT(*)::BIGINT AS issues,
       COUNT(DISTINCT doc_id)::BIGINT AS documents,
       COUNT(DISTINCT table_id)::BIGINT AS tables,
       COUNT(DISTINCT cell_id)::BIGINT AS cells,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share_pct
FROM issues
GROUP BY code
ORDER BY issues DESC
''')
display(issue_by_code)

plt.figure(figsize=(10, 5))
sns.barplot(data=issue_by_code, y="code", x="issues", color="#E45756")
plt.xscale("log")
plt.title("Số issue theo mã (thang log)")
plt.xlabel("Số issue"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
issue_by_field = sql('''
SELECT field, COUNT(*)::BIGINT AS issues,
       COUNT(DISTINCT cell_id)::BIGINT AS affected_cells
FROM issues
GROUP BY field
ORDER BY issues DESC
''')
display(issue_by_field)

issue_matrix = sql('''
SELECT code, field, COUNT(*)::BIGINT AS n
FROM issues
GROUP BY code, field
ORDER BY code, n DESC
''').pivot(index="code", columns="field", values="n").fillna(0).astype("int64")
display(issue_matrix)

In [ ]:
cell_count = int(manifest.get("cell_count", 0))
issue_totals = sql('''
SELECT COUNT(*)::BIGINT AS issues,
       COUNT(DISTINCT cell_id)::BIGINT AS distinct_affected_cells,
       COUNT(DISTINCT table_id)::BIGINT AS affected_tables,
       COUNT(DISTINCT doc_id)::BIGINT AS affected_documents
FROM issues
''').iloc[0]

rates = pd.DataFrame([{
    "tổng_ô_manifest": cell_count,
    "số_issue": int(issue_totals["issues"]),
    "ô_có_issue": int(issue_totals["distinct_affected_cells"]),
    "issue_trên_100_ô": round(100 * issue_totals["issues"] / cell_count, 2) if cell_count else np.nan,
    "ô_có_issue_%": round(100 * issue_totals["distinct_affected_cells"] / cell_count, 2) if cell_count else np.nan,
}])
display(rates)

In [ ]:
top_raw_values = sql('''
WITH ranked AS (
  SELECT code, raw_value, COUNT(*)::BIGINT AS n,
         ROW_NUMBER() OVER (PARTITION BY code ORDER BY COUNT(*) DESC, raw_value) AS rn
  FROM issues
  GROUP BY code, raw_value
)
SELECT code, raw_value, n
FROM ranked
WHERE rn <= 10
ORDER BY code, n DESC
''')
display(top_raw_values)

In [ ]:
issues_per_document = sql('''
SELECT d.doc_id, d.company_code, d.report_year, d.statement_scope,
       COUNT(i.code)::BIGINT AS issues,
       COUNT(DISTINCT i.cell_id)::BIGINT AS affected_cells,
       COUNT(DISTINCT i.table_id)::BIGINT AS affected_tables
FROM documents d
LEFT JOIN issues i USING (doc_id)
GROUP BY d.doc_id, d.company_code, d.report_year, d.statement_scope
ORDER BY issues DESC
''')
display(issues_per_document.head(20))
display(issues_per_document[["issues", "affected_cells", "affected_tables"]].describe(percentiles=[.5,.75,.9,.95,.99]).round(2))

## 9. Kiểm tra khóa, trùng lặp và toàn vẹn tham chiếu

In [ ]:
integrity_queries = {
    "documents.doc_id trùng": "SELECT COUNT(*) FROM (SELECT doc_id FROM documents GROUP BY doc_id HAVING COUNT(*) > 1)",
    "documents.relative_path trùng": "SELECT COUNT(*) FROM (SELECT relative_path FROM documents GROUP BY relative_path HAVING COUNT(*) > 1)",
    "tables.table_id trùng": "SELECT COUNT(*) FROM (SELECT table_id FROM tables GROUP BY table_id HAVING COUNT(*) > 1)",
    "occurrences.source_table_id trùng": "SELECT COUNT(*) FROM (SELECT source_table_id FROM occurrences GROUP BY source_table_id HAVING COUNT(*) > 1)",
    "tables.doc_id mồ côi": "SELECT COUNT(*) FROM tables t LEFT JOIN documents d USING(doc_id) WHERE d.doc_id IS NULL",
    "occurrences.doc_id mồ côi": "SELECT COUNT(*) FROM occurrences o LEFT JOIN documents d USING(doc_id) WHERE d.doc_id IS NULL",
    "issues.doc_id mồ côi": "SELECT COUNT(*) FROM issues i LEFT JOIN documents d USING(doc_id) WHERE d.doc_id IS NULL",
    "issues.table_id mồ côi": "SELECT COUNT(*) FROM issues i LEFT JOIN tables t USING(table_id) WHERE t.table_id IS NULL",
    "canonical_table_id mồ côi": '''SELECT COUNT(*) FROM occurrences o LEFT JOIN tables t
        ON o.canonical_table_id = t.table_id
        WHERE o.canonical_table_id IS NOT NULL AND t.table_id IS NULL''',
    "line range bảng không hợp lệ": "SELECT COUNT(*) FROM tables WHERE line_end < line_start",
    "line range occurrence không hợp lệ": "SELECT COUNT(*) FROM occurrences WHERE line_end < line_start",
    "kích thước bảng không dương": "SELECT COUNT(*) FROM tables WHERE row_count <= 0 OR column_count <= 0",
}
integrity = pd.DataFrame([
    {"kiểm_tra": name, "số_vi_phạm": int(sql(query).iloc[0, 0])}
    for name, query in integrity_queries.items()
])
integrity["đạt"] = integrity["số_vi_phạm"].eq(0)
display(integrity)

In [ ]:
canonical_multiplicity = sql('''
SELECT canonical_table_id, COUNT(*)::BIGINT AS source_occurrences
FROM occurrences
WHERE canonical_table_id IS NOT NULL
GROUP BY canonical_table_id
HAVING COUNT(*) > 1
ORDER BY source_occurrences DESC, canonical_table_id
''')
display(Markdown(f"Canonical table có nhiều hơn một source occurrence: **{len(canonical_multiplicity):,}**"))
display(canonical_multiplicity.head(20))

## 10. Truy vết provenance mẫu

Ví dụ dưới đây nối chuỗi `issue → bảng chuẩn → lần xuất hiện nguồn → tài liệu`, đúng định hướng kiểm toán được của dự án. Có thể thay `SAMPLE_CODE` để xem một loại issue khác.

In [ ]:
SAMPLE_CODE = "statement_conflict"
provenance_sample = sql('''
SELECT i.code, i.field, i.raw_value, i.cell_id,
       t.table_id, t.title_raw, t.statement_type, t.unit_normalized,
       o.relative_path, o.line_start, o.line_end, o.status,
       d.company_code, d.report_year, d.statement_scope, d.sha256
FROM issues i
JOIN tables t USING (table_id, doc_id)
LEFT JOIN occurrences o
  ON o.canonical_table_id = t.table_id AND o.doc_id = t.doc_id
JOIN documents d USING (doc_id)
WHERE i.code = ?
ORDER BY d.company_code, d.report_year, t.table_id
LIMIT 25
''', [SAMPLE_CODE])
display(provenance_sample)

## 11. Tổng hợp phát hiện tự động

In [ ]:
n_docs = int(doc_summary.loc[0, "documents"])
n_tables = int(table_summary.loc[0, "tables"])
n_issues = int(issue_totals["issues"])
affected_cells = int(issue_totals["distinct_affected_cells"])
missing_stmt = int(table_meta.loc[0, "missing_statement_type"])
missing_unit = int(table_meta.loc[0, "missing_unit_normalized"])
canonical = int(status_actual.get("canonical", 0))
rejected = int(status_actual.get("rejected", 0))
duplicate = int(status_actual.get("duplicate", 0))
violations = int(integrity["số_vi_phạm"].sum())
top_issue = issue_by_code.iloc[0]

summary_md = f'''
### Kết luận chính

- Bộ dữ liệu có **{n_docs:,} tài liệu**, **{n_tables:,} bảng chuẩn hóa** và **{n_issues:,} issue**.
- Manifest ghi nhận **{cell_count:,} ô**; **{affected_cells:,} ô** xuất hiện trong bảng issue, tương đương **{100*affected_cells/cell_count:.2f}%** nếu mỗi `cell_id` tương ứng một ô duy nhất.
- Issue lớn nhất là **`{top_issue['code']}`** với **{int(top_issue['issues']):,}** bản ghi ({float(top_issue['share_pct']):.2f}% tổng issue).
- Metadata bảng còn thiếu nhiều: `statement_type` thiếu **{missing_stmt:,}** bảng ({100*missing_stmt/n_tables:.2f}%) và `unit_normalized` thiếu **{missing_unit:,}** bảng ({100*missing_unit/n_tables:.2f}%).
- Pipeline occurrence tạo **{canonical:,} canonical**, loại **{rejected:,} rejected** và đánh dấu **{duplicate:,} duplicate**.
- Tổng số vi phạm trong bộ kiểm tra khóa/tham chiếu/range ở trên là **{violations:,}**. Xem từng dòng để phân biệt lỗi khóa nghiêm trọng với quy tắc miền dữ liệu.

### Ưu tiên xử lý đề xuất

1. Ưu tiên cải thiện nhận diện **đơn vị**, vì `unit_unknown` chiếm phần lớn issue và `unit_normalized` còn thiếu ở đa số bảng.
2. Cải thiện suy luận **kỳ báo cáo** cho `period_incomplete`, nhưng giữ cơ chế từ chối có kiểm soát khi không đủ bằng chứng.
3. Bổ sung classifier/alias cho `statement_type` và metric; đo lại theo công ty, năm và scope để tránh cải thiện cục bộ.
4. Duy trì các kiểm tra toàn vẹn tham chiếu trong CI, cùng đối chiếu fingerprint và manifest sau mỗi lần rebuild.
5. Luôn giữ provenance `issue/cell → table → source occurrence → document` trong mọi báo cáo đánh giá.
'''
display(Markdown(summary_md))

## 12. Hàm drill-down tái sử dụng

Dùng hàm dưới đây để điều tra một công ty, một tài liệu hoặc một bảng cụ thể mà không sửa các truy vấn phía trên.

In [ ]:
def inspect_company(company_code: str, limit: int = 100):
    '''Trả về tài liệu, bảng và thống kê issue của một mã công ty.'''
    docs = sql("SELECT * FROM documents WHERE company_code = ? ORDER BY report_year, statement_scope", [company_code])
    tables_ = sql('''
        SELECT t.*, d.report_year, d.statement_scope
        FROM tables t JOIN documents d USING(doc_id)
        WHERE d.company_code = ?
        ORDER BY d.report_year, t.line_start
        LIMIT ?
    ''', [company_code, limit])
    issues_ = sql('''
        SELECT d.report_year, i.code, COUNT(*)::BIGINT AS n
        FROM issues i JOIN documents d USING(doc_id)
        WHERE d.company_code = ?
        GROUP BY d.report_year, i.code
        ORDER BY d.report_year, n DESC
    ''', [company_code])
    return {"documents": docs, "tables": tables_, "issues_by_year": issues_}

def inspect_table(table_id: str, issue_limit: int = 200):
    '''Trả về metadata, occurrence và issue của một table_id.'''
    metadata = sql('''
        SELECT t.*, d.company_code, d.report_year, d.statement_scope, d.relative_path
        FROM tables t JOIN documents d USING(doc_id)
        WHERE t.table_id = ?
    ''', [table_id])
    occurrences_ = sql("SELECT * FROM occurrences WHERE canonical_table_id = ? ORDER BY ordinal", [table_id])
    issues_ = sql("SELECT * FROM issues WHERE table_id = ? LIMIT ?", [table_id, issue_limit])
    return {"metadata": metadata, "occurrences": occurrences_, "issues": issues_}

print("Sẵn sàng: inspect_company('AAA') hoặc inspect_table('<table_id>')")